In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container_raw", "raw")
dbutils.widgets.text("catalog", "catalog_dev")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_golden", "golden")
dbutils.widgets.text("storageName", "adlsproyectofinaljctfd01")
dbutils.widgets.dropdown("confirmar", "NO", ["NO", "SI"], "Confirmar eliminación")

In [0]:
container_raw = dbutils.widgets.get("container_raw")
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_golden = dbutils.widgets.get("schema_golden")
storageName = dbutils.widgets.get("storageName")
confirmar = dbutils.widgets.get("confirmar")

In [0]:
if confirmar != "SI":
    dbutils.notebook.exit("Cancelado: cambia el widget 'confirmar' a SI para ejecutar los drops.")

TABLAS = {
    schema_golden: ["kpi_vuelos", "kpi_hoteles", "kpi_clientes", "kpi_destinos", "kpi_ventas_mensuales",
                    "fact_reservas", "dim_fecha", "dim_vuelo", "dim_hotel", "dim_destino", "dim_cliente"],
    schema_silver: ["reservas_enriquecidas", "pagos", "reservas", "vuelos", "hoteles", "clientes"],
    schema_bronze: ["pagos", "reservas", "vuelos", "hoteles", "clientes"],
}

for esquema, tablas in TABLAS.items():
    for tabla in tablas:
        spark.sql(f"DROP TABLE IF EXISTS {catalog}.{esquema}.{tabla}")
        print(f"Tabla eliminada: {catalog}.{esquema}.{tabla}")

Tabla eliminada: catalog_dev.golden.kpi_vuelos
Tabla eliminada: catalog_dev.golden.kpi_hoteles
Tabla eliminada: catalog_dev.golden.kpi_clientes
Tabla eliminada: catalog_dev.golden.kpi_destinos
Tabla eliminada: catalog_dev.golden.kpi_ventas_mensuales
Tabla eliminada: catalog_dev.golden.fact_reservas
Tabla eliminada: catalog_dev.golden.dim_fecha
Tabla eliminada: catalog_dev.golden.dim_vuelo
Tabla eliminada: catalog_dev.golden.dim_hotel
Tabla eliminada: catalog_dev.golden.dim_destino
Tabla eliminada: catalog_dev.golden.dim_cliente
Tabla eliminada: catalog_dev.silver.reservas_enriquecidas
Tabla eliminada: catalog_dev.silver.pagos
Tabla eliminada: catalog_dev.silver.reservas
Tabla eliminada: catalog_dev.silver.vuelos
Tabla eliminada: catalog_dev.silver.hoteles
Tabla eliminada: catalog_dev.silver.clientes
Tabla eliminada: catalog_dev.bronze.pagos
Tabla eliminada: catalog_dev.bronze.reservas
Tabla eliminada: catalog_dev.bronze.vuelos
Tabla eliminada: catalog_dev.bronze.hoteles
Tabla eliminada

In [0]:
for esquema in [schema_bronze, schema_silver, schema_golden]:
    ruta = f"abfss://{esquema}@{storageName}.dfs.core.windows.net/"
    dbutils.fs.rm(ruta, True)
    print(f"Archivos eliminados: {ruta}")

Archivos eliminados: abfss://bronze@adlsproyectofinaljctfd01.dfs.core.windows.net/
Archivos eliminados: abfss://silver@adlsproyectofinaljctfd01.dfs.core.windows.net/
Archivos eliminados: abfss://golden@adlsproyectofinaljctfd01.dfs.core.windows.net/


In [0]:
for esquema in [container_raw, schema_bronze, schema_silver, schema_golden]:
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{esquema} CASCADE")
    print(f"Esquema eliminado: {catalog}.{esquema}")

Esquema eliminado: catalog_dev.raw
Esquema eliminado: catalog_dev.bronze
Esquema eliminado: catalog_dev.silver
Esquema eliminado: catalog_dev.golden


In [0]:
spark.sql(f"DROP CATALOG IF EXISTS {catalog} CASCADE")
print(f"Catálogo eliminado: {catalog}")

Catálogo eliminado: catalog_dev


In [0]:
for ubicacion in ["exlt-golden", "exlt-silver", "exlt-bronze", "exlt-raw", "exlt-metastore"]:
    spark.sql(f"DROP EXTERNAL LOCATION IF EXISTS `{ubicacion}`")
    print(f"External location eliminada: {ubicacion}")

External location eliminada: exlt-golden
External location eliminada: exlt-silver
External location eliminada: exlt-bronze
External location eliminada: exlt-raw
External location eliminada: exlt-metastore
